In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from IPython.display import display

print("Library berhasil diimpor")

Library berhasil diimpor


In [3]:
hasil_k5 = pd.read_csv("hasil_prediksi_k5.csv")
hasil_k10 = pd.read_csv("hasil_prediksi_k10.csv")
hasil_k20 = pd.read_csv("hasil_prediksi_k20.csv")

print("Hasil prediksi berhasil dibaca")

Hasil prediksi berhasil dibaca


In [4]:
ringkasan_prediksi = pd.DataFrame({
    "K": [5, 10, 20],
    "jumlah_data": [
        len(hasil_k5),
        len(hasil_k10),
        len(hasil_k20)
    ],
    "prediksi_kosong": [
        hasil_k5["rating_prediksi"].isnull().sum(),
        hasil_k10["rating_prediksi"].isnull().sum(),
        hasil_k20["rating_prediksi"].isnull().sum()
    ]
})

display(ringkasan_prediksi)

,K,jumlah_data,prediksi_kosong
0,5,20417,0
1,10,20417,0
2,20,20417,0


In [ ]:
def hitung_rmse(data):
    nilai_rmse = np.sqrt(
        mean_squared_error(
            data["rating"],
            data["rating_prediksi"]
        )
    )

    return nilai_rmse

In [6]:
rmse_k5 = hitung_rmse(hasil_k5)
rmse_k10 = hitung_rmse(hasil_k10)
rmse_k20 = hitung_rmse(hasil_k20)

hasil_rmse = pd.DataFrame({
    "K": [5, 10, 20],
    "RMSE": [
        rmse_k5,
        rmse_k10,
        rmse_k20
    ]
})

hasil_rmse = hasil_rmse.sort_values(
    "RMSE"
).reset_index(drop=True)

hasil_rmse["RMSE"] = hasil_rmse["RMSE"].round(4)

display(hasil_rmse)

,K,RMSE
0,20,0.9607
1,10,0.9661
2,5,0.9894


In [7]:
def hitung_precision_10(data):
    nilai_precision = []

    for user_id, data_user in data.groupby("userId"):
        top_10 = data_user.sort_values(
            "rating_prediksi",
            ascending=False
        ).head(10)

        jumlah_relevan = (
            top_10["rating"] >= 4.0
        ).sum()

        precision_user = jumlah_relevan / len(top_10)

        nilai_precision.append(precision_user)

    return np.mean(nilai_precision)

In [8]:
precision_k5 = hitung_precision_10(hasil_k5)
precision_k10 = hitung_precision_10(hasil_k10)
precision_k20 = hitung_precision_10(hasil_k20)

hasil_precision = pd.DataFrame({
    "K": [5, 10, 20],
    "Precision@10": [
        precision_k5,
        precision_k10,
        precision_k20
    ]
})

hasil_precision = hasil_precision.sort_values(
    "Precision@10",
    ascending=False
).reset_index(drop=True)

hasil_precision["Precision@10"] = hasil_precision[
    "Precision@10"
].round(4)

display(hasil_precision)

,K,Precision@10
0,20,0.6561
1,10,0.6540
2,5,0.6507


In [9]:
perbandingan_evaluasi = hasil_rmse.merge(
    hasil_precision,
    on="K",
    how="inner"
)

perbandingan_evaluasi = perbandingan_evaluasi.sort_values(
    "RMSE"
).reset_index(drop=True)

display(perbandingan_evaluasi)

,K,RMSE,Precision@10
0,20,0.9607,0.6561
1,10,0.9661,0.6540
2,5,0.9894,0.6507


In [10]:
k_terbaik_rmse = hasil_rmse.loc[
    hasil_rmse["RMSE"].idxmin(),
    "K"
]

k_terbaik_precision = hasil_precision.loc[
    hasil_precision["Precision@10"].idxmax(),
    "K"
]

print("K terbaik berdasarkan RMSE:", k_terbaik_rmse)
print("K terbaik berdasarkan Precision@10:", k_terbaik_precision)

K terbaik berdasarkan RMSE: 20
K terbaik berdasarkan Precision@10: 20


In [11]:
perbandingan_evaluasi.to_csv(
    "hasil_evaluasi.csv",
    index=False
)

hasil_terbaik = perbandingan_evaluasi[
    perbandingan_evaluasi["K"] == 20
].copy()

hasil_terbaik.to_csv(
    "hasil_evaluasi_k_terbaik.csv",
    index=False
)

print("Hasil evaluation berhasil disimpan.")
display(hasil_terbaik)

Hasil evaluation berhasil disimpan.


,K,RMSE,Precision@10
0,20,0.9607,0.6561
